# OCR Financial Reports — Dataset Profile

Notebook này phân tích **chỉ đọc** cây `ocr_result`: cấu trúc thư mục, độ phủ công ty–năm, loại báo cáo, kích thước file và chất lượng một mẫu nội dung OCR. Kết quả dùng để quyết định cách xây dựng inventory, ingestion và normalization; notebook không sửa dữ liệu gốc.

> Chạy **Run All** sau khi kiểm tra `DATA_ROOT` ở cell cấu hình.

## 0. Configuration

Mặc định notebook tìm dataset tại `data/raw/ocr_annual_financials/ocr_result`. Có thể đổi `DATA_ROOT` nếu bạn lưu dữ liệu ở vị trí khác. Phần metadata quét toàn bộ TXT; phần nội dung chỉ đọc một mẫu cố định để tránh I/O quá lớn.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "pyproject.toml").exists()), Path.cwd().resolve()
)
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "ocr_annual_financials" / "ocr_result"
CONTENT_SAMPLE_SIZE = 500
RANDOM_SEED = 42
MAX_CONTENT_BYTES = 2_000_000
EXPECTED_YEAR_RANGE = range(2015, 2026)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset root : {DATA_ROOT}")
print(f"Content sample: at most {CONTENT_SAMPLE_SIZE:,} files")

In [ ]:
import random
import re
import unicodedata
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import pandas as pd

INVENTORY_COLUMNS = [
    "path",
    "relative_path",
    "ticker",
    "year",
    "year_in_expected_range",
    "report_name",
    "scope",
    "assurance",
    "period_type",
    "structure_status",
    "structure_issue",
    "size_bytes",
    "is_empty",
    "is_tiny",
    "modified_at",
    "stat_error",
]


def _normalize_label(value: str) -> str:
    decomposed = unicodedata.normalize("NFKD", value)
    ascii_like = "".join(char for char in decomposed if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "", ascii_like.casefold())


def parse_report_path(path: Path, root: Path) -> dict[str, object]:
    """Parse hierarchy metadata without assuming the report-name tokens are complete."""
    try:
        relative = path.relative_to(root)
    except ValueError:
        relative = path
    parts = relative.parts
    ticker_raw = parts[0] if len(parts) >= 1 else ""
    year_raw = parts[1] if len(parts) >= 2 else ""
    report_name = parts[-2] if len(parts) >= 4 else ""
    ticker = ticker_raw.upper() if re.fullmatch(r"[A-Za-z0-9]{2,10}", ticker_raw) else None
    year = int(year_raw) if re.fullmatch(r"\d{4}", year_raw) else None
    normalized = _normalize_label(report_name)

    if "hopnhat" in normalized:
        scope = "consolidated"
    elif "congtyme" in normalized or "riengle" in normalized or "rieng" in normalized:
        scope = "separate"
    else:
        scope = "unspecified"

    if "kiemtoan" in normalized:
        assurance = "audited"
    elif "soatxet" in normalized:
        assurance = "reviewed"
    else:
        assurance = "unspecified"

    if re.search(r"quy[1-4]", normalized):
        period_type = "quarterly"
    elif "bannien" in normalized or "6thang" in normalized:
        period_type = "semiannual"
    else:
        period_type = "annual_or_unspecified"

    issues = []
    if len(parts) < 4:
        issues.append("expected ticker/year/report/file hierarchy")
    if ticker is None:
        issues.append("invalid ticker directory")
    if year is None or not 1900 <= year <= 2100:
        issues.append("invalid year directory")

    return {
        "path": path,
        "relative_path": relative.as_posix(),
        "ticker": ticker,
        "year": year,
        "year_in_expected_range": year in range(2015, 2026) if year is not None else False,
        "report_name": report_name or None,
        "scope": scope,
        "assurance": assurance,
        "period_type": period_type,
        "structure_status": "valid" if not issues else "malformed",
        "structure_issue": "; ".join(issues) if issues else None,
    }


def build_inventory(root: Path) -> pd.DataFrame:
    """Collect deterministic read-only metadata for every TXT file below root."""
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {root}")
    records: list[dict[str, Any]] = []
    for path in sorted(root.rglob("*.txt"), key=lambda item: item.as_posix().casefold()):
        record: dict[str, Any] = parse_report_path(path, root)
        try:
            stat = path.stat()
            record.update(
                size_bytes=stat.st_size,
                is_empty=stat.st_size == 0,
                is_tiny=stat.st_size < 1024,
                modified_at=pd.Timestamp(stat.st_mtime, unit="s", tz="UTC"),
                stat_error=None,
            )
        except OSError as error:
            record.update(
                size_bytes=None,
                is_empty=False,
                is_tiny=False,
                modified_at=pd.NaT,
                stat_error=str(error),
            )
        records.append(record)
    return pd.DataFrame.from_records(records, columns=INVENTORY_COLUMNS)


def sample_paths(paths: Sequence[Path], sample_size: int, seed: int) -> list[Path]:
    """Return a stable bounded sample independent of input ordering."""
    if sample_size < 0:
        raise ValueError("sample_size must not be negative")
    ordered = sorted((Path(path) for path in paths), key=lambda item: item.as_posix().casefold())
    if sample_size >= len(ordered):
        return ordered
    return sorted(
        random.Random(seed).sample(ordered, sample_size),
        key=lambda item: item.as_posix().casefold(),
    )


def inspect_text_file(path: Path, max_bytes: int) -> dict[str, object]:
    """Inspect bounded bytes and preserve decoding failures as quality metrics."""
    base: dict[str, object] = {
        "path": path,
        "bytes_read": 0,
        "truncated": False,
        "utf8_valid": False,
        "read_error": None,
        "line_count": 0,
        "character_count": 0,
        "replacement_char_count": 0,
        "replacement_ratio": 0.0,
        "control_char_count": 0,
        "numeric_ratio": 0.0,
        "has_html_table": False,
        "has_pipe_table": False,
        "has_tabular_markers": False,
    }
    if max_bytes < 1:
        raise ValueError("max_bytes must be at least 1")
    try:
        with path.open("rb") as stream:
            payload = stream.read(max_bytes + 1)
    except OSError as error:
        base["read_error"] = str(error)
        return base

    truncated = len(payload) > max_bytes
    payload = payload[:max_bytes]
    try:
        text = payload.decode("utf-8", errors="strict")
        utf8_valid = True
    except UnicodeDecodeError:
        text = payload.decode("utf-8", errors="replace")
        utf8_valid = False

    lowered = text.casefold()
    lines = text.splitlines()
    replacement_count = text.count("�")
    non_whitespace = [char for char in text if not char.isspace()]
    pipe_lines = sum(line.count("|") >= 2 for line in lines)
    html_table = "<table" in lowered or ("<tr" in lowered and "<td" in lowered)
    has_pipe_table = pipe_lines > 0
    denominator = max(len(text), 1)
    numeric_denominator = max(len(non_whitespace), 1)
    base.update(
        bytes_read=len(payload),
        truncated=truncated,
        utf8_valid=utf8_valid,
        line_count=len(lines),
        character_count=len(text),
        replacement_char_count=replacement_count,
        replacement_ratio=replacement_count / denominator,
        control_char_count=sum(ord(char) < 32 and char not in "\n\r\t" for char in text),
        numeric_ratio=sum(char.isdigit() for char in non_whitespace) / numeric_denominator,
        has_html_table=html_table,
        has_pipe_table=has_pipe_table,
        has_tabular_markers=html_table or has_pipe_table or "\t" in text,
    )
    return base


def build_readiness_summary(inventory: pd.DataFrame, content: pd.DataFrame) -> pd.DataFrame:
    """Map observed corpus risks to the next concrete engineering actions."""
    total = len(inventory)
    malformed = int(
        (inventory.get("structure_status", pd.Series(dtype="object")) == "malformed").sum()
    )
    empty = int(inventory.get("is_empty", pd.Series(dtype="bool")).fillna(False).sum())
    read_errors = int(content.get("read_error", pd.Series(dtype="object")).notna().sum())
    invalid_utf8 = int((content.get("utf8_valid", pd.Series(dtype="bool")) == False).sum())  # noqa: E712
    table_markers = (
        float(content.get("has_tabular_markers", pd.Series(dtype="bool")).mean())
        if len(content)
        else 0.0
    )
    valid_pairs = (
        inventory.dropna(subset=["ticker", "year"])
        if {"ticker", "year"} <= set(inventory.columns)
        else pd.DataFrame()
    )
    rows = [
        {
            "priority": "P0",
            "finding": "path parsing",
            "evidence": f"{malformed:,}/{total:,} malformed paths",
            "next_action": (
                "Keep tolerant parsing and record structure_issue in the immutable inventory."
            ),
        },
        {
            "priority": "P0",
            "finding": "quarantine",
            "evidence": f"{empty:,} empty files; {read_errors:,} sampled read errors",
            "next_action": (
                "Route unreadable and empty files to logical quarantine without changing raw data."
            ),
        },
        {
            "priority": "P1",
            "finding": "encoding",
            "evidence": f"{invalid_utf8:,}/{len(content):,} sampled files are not strict UTF-8",
            "next_action": "Preserve raw bytes and record the decoder/fallback used per document.",
        },
        {
            "priority": "P1",
            "finding": "table detection",
            "evidence": f"{table_markers:.1%} of sampled files contain simple table markers",
            "next_action": (
                "Combine HTML/line heuristics with a fallback table detector; "
                "do not rely on one marker."
            ),
        },
        {
            "priority": "P2",
            "finding": "coverage",
            "evidence": (
                f"{len(valid_pairs.drop_duplicates(subset=['ticker', 'year'])):,} "
                "observed ticker-year pairs"
            ),
            "next_action": (
                "Make retrieval missing-year aware and never infer that "
                "an absent report equals a zero value."
            ),
        },
    ]
    return pd.DataFrame(rows, columns=["priority", "finding", "evidence", "next_action"])

## 1. Synthetic self-check

Cell này kiểm tra nhanh parser và inventory trên dữ liệu tạm, không ghi vào dataset thật.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as temporary_directory:
    synthetic_root = Path(temporary_directory) / "ocr_result"
    valid_file = (
        synthetic_root
        / "HTG"
        / "2022"
        / "HTG_Baocaotaichinh_2022_Kiemtoan_Hopnhat"
        / "report_extracted.txt"
    )
    malformed_file = synthetic_root / "loose.txt"
    valid_file.parent.mkdir(parents=True)
    valid_file.write_text("<table><tr><td>100</td></tr></table>", encoding="utf-8")
    malformed_file.write_text("OCR noise", encoding="utf-8")
    synthetic_inventory = build_inventory(synthetic_root)
    assert set(synthetic_inventory["structure_status"]) == {"valid", "malformed"}
    assert inspect_text_file(valid_file, 10_000)["has_html_table"] is True

print("Self-check passed")

## 2. Corpus overview

Quét metadata của toàn bộ file TXT. Cell này không đọc nội dung file.

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy ocr_result tại {DATA_ROOT}. "
        "Hãy sửa DATA_ROOT hoặc chạy: bash scripts/download_dataset.sh --download"
    )

inventory = build_inventory(DATA_ROOT)
if inventory.empty:
    raise RuntimeError(
        f"Không tìm thấy file TXT dưới {DATA_ROOT}. "
        "Cấu trúc mong đợi: ocr_result/<TICKER>/<YEAR>/<REPORT>/*.txt"
    )

valid_inventory = inventory[inventory["structure_status"].eq("valid")].copy()
valid_years = valid_inventory["year"].dropna().astype(int)
kpis = pd.DataFrame(
    [
        ("TXT files", f"{len(inventory):,}"),
        ("Parsed tickers", f"{valid_inventory['ticker'].nunique():,}"),
        (
            "Observed years",
            f"{valid_years.min()}–{valid_years.max()}" if len(valid_years) else "n/a",
        ),
        ("Total size", f"{inventory['size_bytes'].fillna(0).sum() / 1024**3:,.2f} GiB"),
        ("Malformed paths", f"{inventory['structure_status'].eq('malformed').sum():,}"),
        ("Empty files", f"{inventory['is_empty'].fillna(False).sum():,}"),
        ("Median file size", f"{inventory['size_bytes'].median() / 1024:,.1f} KiB"),
    ],
    columns=["Metric", "Value"],
)
display(kpis.style.hide(axis="index").set_properties(**{"text-align": "left"}))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

reports_by_year = valid_inventory["year"].value_counts().sort_index()
reports_by_year.plot(kind="bar", ax=axes[0, 0], color="#2563eb", title="Reports by year")
axes[0, 0].set(xlabel="Year", ylabel="TXT files")

top_tickers = valid_inventory["ticker"].value_counts().head(20).sort_values()
top_tickers.plot(kind="barh", ax=axes[0, 1], color="#0f766e", title="Top 20 tickers by reports")
axes[0, 1].set(xlabel="TXT files", ylabel="Ticker")

scope_counts = valid_inventory["scope"].value_counts().sort_values()
scope_counts.plot(
    kind="barh", ax=axes[1, 0], color="#7c3aed", title="Statement scope inferred from names"
)
axes[1, 0].set(xlabel="TXT files", ylabel="Scope")

positive_sizes_mib = inventory.loc[inventory["size_bytes"].gt(0), "size_bytes"] / 1024**2
axes[1, 1].hist(positive_sizes_mib, bins=50, color="#ea580c", alpha=0.85)
if len(positive_sizes_mib) and positive_sizes_mib.max() > max(positive_sizes_mib.min(), 0.001) * 10:
    axes[1, 1].set_xscale("log")
axes[1, 1].set(title="File-size distribution", xlabel="MiB (log scale when useful)", ylabel="Files")

fig.suptitle("OCR corpus at a glance", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

## 3. Coverage

Heatmap cho 40 mã có nhiều báo cáo nhất. Ô càng đậm nghĩa là mã đó có nhiều file trong năm tương ứng; ô trắng không đồng nghĩa giá trị tài chính bằng 0.

In [ ]:
top_coverage_tickers = valid_inventory["ticker"].value_counts().head(40).index
coverage = (
    valid_inventory[valid_inventory["ticker"].isin(top_coverage_tickers)]
    .pivot_table(
        index="ticker", columns="year", values="relative_path", aggfunc="size", fill_value=0
    )
    .reindex(index=top_coverage_tickers, columns=list(EXPECTED_YEAR_RANGE), fill_value=0)
)

fig, ax = plt.subplots(figsize=(15, max(6, len(coverage) * 0.28)))
image = ax.imshow(coverage.to_numpy(), aspect="auto", cmap="Blues")
ax.set_xticks(
    range(len(coverage.columns)), labels=[str(year) for year in coverage.columns], rotation=45
)
ax.set_yticks(range(len(coverage.index)), labels=coverage.index)
ax.set(title="Ticker–year report coverage (top 40 tickers)", xlabel="Year", ylabel="Ticker")
fig.colorbar(image, ax=ax, label="TXT files")
fig.tight_layout()
plt.show()

coverage_summary = pd.DataFrame(
    {
        "reports": valid_inventory.groupby("ticker").size(),
        "years_present": valid_inventory.groupby("ticker")["year"].nunique(),
        "first_year": valid_inventory.groupby("ticker")["year"].min(),
        "last_year": valid_inventory.groupby("ticker")["year"].max(),
    }
).sort_values(["years_present", "reports"], ascending=False)
display(coverage_summary.head(50))

## 4. Anomalies

Các bảng dưới đây là ứng viên cần parser chịu lỗi hoặc logical quarantine. Chúng chỉ là tín hiệu kiểm tra, không tự động kết luận file phải bị xóa.

In [ ]:
malformed_paths = inventory[inventory["structure_status"].eq("malformed")]
tiny_files = inventory[inventory["size_bytes"].fillna(-1).between(0, 1023)]
size_threshold = inventory["size_bytes"].dropna().quantile(0.995)
large_files = inventory[inventory["size_bytes"].gt(size_threshold)]
duplicate_mask = valid_inventory.duplicated(["ticker", "year", "report_name"], keep=False)
duplicate_reports = valid_inventory[duplicate_mask].sort_values(["ticker", "year", "report_name"])
outside_expected_years = valid_inventory[~valid_inventory["year_in_expected_range"]]

anomaly_summary = pd.DataFrame(
    {
        "Anomaly": [
            "Malformed hierarchy",
            "Empty/tiny (<1 KiB)",
            "Top 0.5% by size",
            "Duplicate report keys",
            "Outside 2015–2025",
        ],
        "Files": [
            len(malformed_paths),
            len(tiny_files),
            len(large_files),
            len(duplicate_reports),
            len(outside_expected_years),
        ],
    }
)
display(
    anomaly_summary.style.hide(axis="index").background_gradient(subset=["Files"], cmap="Oranges")
)

for title, frame in (
    ("Malformed paths", malformed_paths),
    ("Empty/tiny files", tiny_files),
    ("Unusually large files", large_files),
    ("Duplicate report keys", duplicate_reports),
):
    display(Markdown(f"### {title} — showing up to 30"))
    columns = [
        column
        for column in [
            "ticker",
            "year",
            "report_name",
            "size_bytes",
            "structure_issue",
            "relative_path",
        ]
        if column in frame
    ]
    display(frame[columns].head(30))

## 5. Content quality sample

Đọc tối đa `MAX_CONTENT_BYTES` byte từ một mẫu xác định bởi `RANDOM_SEED`. Đây là phép đo định hướng, không phải kiểm định toàn bộ corpus.

In [ ]:
candidate_paths = valid_inventory.loc[valid_inventory["stat_error"].isna(), "path"].tolist()
selected_paths = sample_paths(candidate_paths, CONTENT_SAMPLE_SIZE, RANDOM_SEED)
content_profile = pd.DataFrame(
    [inspect_text_file(path, MAX_CONTENT_BYTES) for path in selected_paths]
)

if content_profile.empty:
    raise RuntimeError("Không có file hợp lệ để lấy mẫu nội dung.")

content_kpis = pd.DataFrame(
    [
        ("Sampled files", f"{len(content_profile):,}"),
        ("Strict UTF-8", f"{content_profile['utf8_valid'].mean():.1%}"),
        ("Read errors", f"{content_profile['read_error'].notna().sum():,}"),
        ("Truncated at byte cap", f"{content_profile['truncated'].mean():.1%}"),
        ("HTML table markers", f"{content_profile['has_html_table'].mean():.1%}"),
        ("Any simple table marker", f"{content_profile['has_tabular_markers'].mean():.1%}"),
        ("Median lines", f"{content_profile['line_count'].median():,.0f}"),
        ("Median numeric ratio", f"{content_profile['numeric_ratio'].median():.1%}"),
    ],
    columns=["Metric", "Value"],
)
display(content_kpis.style.hide(axis="index"))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].hist(content_profile["line_count"], bins=40, color="#2563eb")
axes[0].set(title="Line count in sample", xlabel="Lines", ylabel="Files")
axes[1].hist(content_profile["numeric_ratio"], bins=30, color="#0f766e")
axes[1].set(title="Numeric-character ratio", xlabel="Ratio", ylabel="Files")
marker_rates = (
    content_profile[["has_html_table", "has_pipe_table", "has_tabular_markers"]]
    .mean()
    .sort_values()
)
marker_rates.plot(kind="barh", ax=axes[2], color="#7c3aed", title="Table-marker coverage")
axes[2].set(xlabel="Share of sampled files", xlim=(0, 1))
fig.tight_layout()
plt.show()

display(Markdown("### Highest replacement-character ratios"))
display(content_profile.sort_values("replacement_ratio", ascending=False).head(30))

## 6. Recommended next steps

Bảng này chuyển các quan sát thành yêu cầu cụ thể cho inventory và ingestion. Ưu tiên P0 nên được xử lý trước khi trích bảng hàng loạt.

In [ ]:
readiness = build_readiness_summary(inventory, content_profile)
display(
    readiness.style.hide(axis="index")
    .set_properties(subset=["next_action"], **{"text-align": "left", "min-width": "420px"})
    .set_properties(subset=["evidence"], **{"text-align": "left", "min-width": "220px"})
)

print(
    "Suggested order: inventory manifest → tolerant TXT reader → provenance "
    "→ table detector → normalization."
)